In [ ]:
DRIVE <- "/content/drive/MyDrive/thesis/dlnm-pilot"
system(sprintf("cd /content && cp %s/r_library.tar.gz . && tar -xzf r_library.tar.gz", DRIVE))

In [ ]:
cat("=== substrates ===\n")
for (nm in c("mtl","van")) {
  s <- get(nm)
  cat(sprintf("%s  n_da: %d  temp_mat: %s  align: %s  NA: %d\n",
              nm, s$n_da, paste(dim(s$temp_mat), collapse="x"),
              all(rownames(s$temp_mat) == s$truth_factors$DAUID), sum(is.na(s$temp_mat))))
}

cat("\n=== functions present ===\n")
for (fn in c("qaic","reduce_fit","fit_stage1","build_crossbasis","build_city_sim_substrate"))
  cat(sprintf("%-26s %s\n", fn, exists(fn)))

cat("\n=== fixed-version fingerprints ===\n")
cat("qaic uses deviance form:", any(grepl("deviance", deparse(body(qaic)))), "\n")
cat("reduce_fit has model.link:", any(grepl("model.link", deparse(body(reduce_fit)))), "\n")

cat("\n=== populations ===\n")
cat("cma_age_data cities:", paste(names(cma_age_data), collapse=", "), "\n")

=== substrates ===


ERROR: Error in get(nm): object 'mtl' not found


In [ ]:
list.files(DRIVE)

[1] "DLNM-Play.ipynb"                "montreal_da_shp.zip"           
 [3] "montreal_daymet_2015_2019.csv"  "r_library.tar.gz"              
 [5] "saves_2026-05-26.tar.gz"        "saves_eod_2026-06-10.tar.gz"   
 [7] "saves_mtlvan_clean.tar.gz"      "saves_pilot_2026-05-27.tar.gz" 
 [9] "saves_pilot_2026-06-03.tar.gz"  "toronto_da_shp.zip"            
[11] "toronto_da.geojson"             "toronto_daymet_2015_2019.csv"  
[13] "vancouver_da_shp.zip"           "vancouver_daymet_2015_2019.csv"

In [ ]:
system(sprintf("cd /content && cp %s/saves_eod_2026-06-10.tar.gz . && tar -xzf saves_eod_2026-06-10.tar.gz", DRIVE))
list.files("/content/saves")

character(0)

In [ ]:
EOD <- "/content/saves_eod"
mtl                    <- readRDS(file.path(EOD, "mtl_substrate.rds"))
van                    <- readRDS(file.path(EOD, "van_substrate.rds"))
qaic                   <- readRDS(file.path(EOD, "fn_qaic.rds"))
reduce_fit             <- readRDS(file.path(EOD, "fn_reduce_fit.rds"))
fit_stage1             <- readRDS(file.path(EOD, "fn_fit_stage1.rds"))
cma_age_data_mtlvan    <- readRDS(file.path(EOD, "cma_age_data_mtlvan.rds"))
cat("loaded:", paste(ls(pattern="mtl|van|qaic|reduce|stage1"), collapse=", "), "\n")

loaded: cma_age_data_mtlvan, fit_stage1, mtl, qaic, reduce_fit, van 


In [ ]:
cat("qaic deviance form:", any(grepl("deviance", deparse(body(qaic)))), "\n")
cat("reduce_fit model.link:", any(grepl("model.link", deparse(body(reduce_fit)))), "\n")
cat("\nmtl n_da:", mtl$n_da, " align:", all(rownames(mtl$temp_mat)==mtl$truth_factors$DAUID), " NA:", sum(is.na(mtl$temp_mat)), "\n")
cat("van n_da:", van$n_da, " align:", all(rownames(van$temp_mat)==van$truth_factors$DAUID), " NA:", sum(is.na(van$temp_mat)), "\n")
cat("pop cities:", paste(names(cma_age_data_mtlvan), collapse=", "), "\n")

qaic deviance form: TRUE 
reduce_fit model.link: TRUE 

mtl n_da: 6504  align: TRUE  NA: 0 
van n_da: 3573  align: TRUE  NA: 0 
pop cities: montreal, vancouver 


In [ ]:
for (o in c("simulate_counts","base_log_rr","lag_weights","study_dates","annual_rates","build_crossbasis"))
  cat(sprintf("%-18s %s\n", o, exists(o)))

simulate_counts    FALSE
base_log_rr        FALSE
lag_weights        FALSE
study_dates        FALSE
annual_rates       FALSE
build_crossbasis   FALSE


In [ ]:
need <- c(
  # DGP machinery (from saves_pilot restore)
  "simulate_counts","base_log_rr","lag_weights","annual_rates","study_dates",
  # cross-basis + fit chain
  "build_crossbasis","fit_stage1","qaic","reduce_fit",
  # substrates + pops (EOD — should be live)
  "mtl","van","cma_age_data_mtlvan",
  # constants
  "warm_months","year_start","year_end","cma_list"
)
for (o in need) cat(sprintf("%-20s %s\n", o, exists(o)))

simulate_counts      FALSE
base_log_rr          FALSE
lag_weights          FALSE
annual_rates         FALSE
study_dates          FALSE
build_crossbasis     FALSE
fit_stage1           TRUE
qaic                 TRUE
reduce_fit           TRUE
mtl                  TRUE
van                  TRUE
cma_age_data_mtlvan  TRUE
warm_months          FALSE
year_start           FALSE
year_end             FALSE
cma_list             FALSE


In [ ]:
system(sprintf("cd /content && cp %s/saves_pilot_2026-06-03.tar.gz . && tar -xzf saves_pilot_2026-06-03.tar.gz", DRIVE))
load("/content/saves/pilot_session.RData")

In [ ]:
qaic       <- readRDS("/content/saves_eod/fn_qaic.rds")
reduce_fit <- readRDS("/content/saves_eod/fn_reduce_fit.rds")

In [ ]:
cat("study_dates len:", length(study_dates), "\n")
cat("qaic deviance form:", any(grepl("deviance", deparse(body(qaic)))), "\n")
cat("reduce_fit model.link:", any(grepl("model.link", deparse(body(reduce_fit)))), "\n")
for (o in c("simulate_counts","base_log_rr","lag_weights","annual_rates","build_crossbasis"))
  cat(sprintf("%-18s %s\n", o, exists(o)))

study_dates len: 765 
qaic deviance form: TRUE 
reduce_fit model.link: TRUE 
simulate_counts    TRUE
base_log_rr        TRUE
lag_weights        TRUE
annual_rates       TRUE
build_crossbasis   TRUE


In [ ]:
set.seed(42)

mmt_mtl <- apply(mtl$temp_mat, 1, function(x) quantile(x, 0.80, na.rm = TRUE))

mtl_da_long <- CJ(da_idx = 1:mtl$n_da, date = study_dates, age_band = names(annual_rates))
pop_long <- melt(mtl$da_age[, .(da_idx, age_0_64, age_65_74, age_75_84, age_85p)],
                 id.vars = "da_idx", variable.name = "age_band", value.name = "pop")
pop_long[, age_band := as.character(age_band)]
mtl_da_long <- pop_long[mtl_da_long, on = c("da_idx","age_band")]
mtl_da_long[, annual_rate := annual_rates[age_band]]
mtl_da_long[, lambda0 := pop * annual_rate / 1000 / 365]

mtl_sim <- simulate_counts(mtl$temp_mat, mtl_da_long, mtl$truth_factors, mmt_mtl, seed = 42)

cat("mtl_sim rows:", nrow(mtl_sim), " NA pop:", sum(is.na(mtl_da_long$pop)),
    " total deaths:", sum(mtl_sim$n_deaths), "\n")
print(mtl_sim[, .(deaths = sum(n_deaths)), by = age_band])

Simulating 6504 DAs × 765 days × 4 age bands
Total deaths simulated: 142934 
Mean deaths per DA-day-age: 0.0072 
% zero days: 99.3 %
mtl_sim rows: 19902240  NA pop: 0  total deaths: 142934 
    age_band deaths
      <char>  <int>
1:  age_0_64   6776
2: age_65_74  32388
3: age_75_84  45847
4:   age_85p  57923


rows: 18,902, 240, exact shape passes. NA pop: 0 , alignment held. Age gradient climbs—85+ has the fewest people but them most deaths in the steep range. If it were flat or inverted, the DGP wired up wrong. 99.3% zero days.. exactly why stage 1 needs cond pois

In [ ]:
library(data.table)


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%




In [ ]:
set.seed(42)
mtl_sliver_idx <- sample(unique(mtl_sim$da_idx), 150)

mtl_sliver <- mtl_sim[da_idx %in% mtl_sliver_idx & age_band == "age_75_84"]
mtl_sliver[, DA_id := da_idx]
setorder(mtl_sliver, da_idx, date)

mtl_temp_long <- data.table(
  da_idx = rep(mtl_sliver_idx, each = 765),
  date   = rep(study_dates, times = 150),
  temp_C = as.vector(t(mtl$temp_mat[mtl_sliver_idx, ]))
)
mtl_sliver <- mtl_temp_long[mtl_sliver, on = c("da_idx","date")]

cb_mtl  <- build_crossbasis(mtl_sliver$temp_C, lag_max = 21)
res_mtl <- fit_stage1(mtl_sliver, cb_mtl, "Montreal", "age_75_84")

cat("Cross-basis:", paste(dim(cb_mtl), collapse=" x "), "\n")
cat("Winner:", res_mtl$winner_variant,
    " qAIC A:", round(res_mtl$qaic_A,1), " qAIC B:", round(res_mtl$qaic_B,1), "\n")
cat("coef:", length(res_mtl$coef), " vcov:", paste(dim(res_mtl$vcov), collapse=" x "),
    " any NA:", any(is.na(res_mtl$coef)), "\n")

ERROR: Error in crossbasis(x = T_series, lag = lag_max, argvar = list(fun = "bs", : could not find function "crossbasis"


In [ ]:
.libPaths(c("/content/site-library", .libPaths()))
suppressPackageStartupMessages({
  library(dlnm); library(gnm); library(mixmeta); library(splines)
  library(sf); library(data.table); library(exactextractr); library(terra)
  library(ggplot2); library(viridis); library(lubridate)
})
cat("crossbasis:", exists("crossbasis"), " gnm:", exists("gnm"),
    " mixmeta:", exists("mixmeta"), " ns:", exists("ns"), "\n")

crossbasis: TRUE  gnm: TRUE  mixmeta: TRUE  ns: TRUE 


In [ ]:
set.seed(42)
mtl_sliver_idx <- sample(unique(mtl_sim$da_idx), 150)

mtl_sliver <- mtl_sim[da_idx %in% mtl_sliver_idx & age_band == "age_75_84"]
mtl_sliver[, DA_id := da_idx]
setorder(mtl_sliver, da_idx, date)

mtl_temp_long <- data.table(
  da_idx = rep(mtl_sliver_idx, each = 765),
  date   = rep(study_dates, times = 150),
  temp_C = as.vector(t(mtl$temp_mat[mtl_sliver_idx, ]))
)
mtl_sliver <- mtl_temp_long[mtl_sliver, on = c("da_idx","date")]

cb_mtl  <- build_crossbasis(mtl_sliver$temp_C, lag_max = 21)
res_mtl <- fit_stage1(mtl_sliver, cb_mtl, "Montreal", "age_75_84")

cat("Cross-basis:", paste(dim(cb_mtl), collapse=" x "), "\n")
cat("Winner:", res_mtl$winner_variant,
    " qAIC A:", round(res_mtl$qaic_A,1), " qAIC B:", round(res_mtl$qaic_B,1), "\n")
cat("coef:", length(res_mtl$coef), " vcov:", paste(dim(res_mtl$vcov), collapse=" x "),
    " any NA:", any(is.na(res_mtl$coef)), "\n")


=== Stage 1: Montreal, age age_75_84 ===
  Variant A: 3750 strata, 114750 rows
  Variant A: FAILED to converge
  Variant B: 26250 strata, 114750 rows
  Variant B: FAILED to converge
  Winner: variant B


ERROR: Error in UseMethod("vcov"): no applicable method for 'vcov' applied to an object of class "NULL"


In [ ]:
dir.create("/content/saves_eod", showWarnings = FALSE)

saveRDS(mtl,                 "/content/saves_eod/mtl_substrate.rds")
saveRDS(van,                 "/content/saves_eod/van_substrate.rds")
saveRDS(qaic,                "/content/saves_eod/fn_qaic.rds")
saveRDS(reduce_fit,          "/content/saves_eod/fn_reduce_fit.rds")
saveRDS(fit_stage1,          "/content/saves_eod/fn_fit_stage1.rds")
saveRDS(cma_age_data_mtlvan, "/content/saves_eod/cma_age_data_mtlvan.rds")
saveRDS(mtl_sim,             "/content/saves_eod/mtl_sim.rds")

today   <- format(Sys.Date(), "%Y-%m-%d")
tarball <- sprintf("/content/saves_eod_%s.tar.gz", today)
system(sprintf("cd /content && tar -czf saves_eod_%s.tar.gz saves_eod/", today))

cat("Saved:", tarball, "—",
    round(file.info(tarball)$size / 1024^2, 2), "MB\n")
cat("Contents:\n"); print(list.files("/content/saves_eod"))

Saved: /content/saves_eod_2026-06-17.tar.gz — 200.56 MB
Contents:
[1] "cma_age_data_mtlvan.rds" "fn_fit_stage1.rds"      
[3] "fn_qaic.rds"             "fn_reduce_fit.rds"      
[5] "mtl_sim.rds"             "mtl_substrate.rds"      
[7] "van_substrate.rds"      


In [ ]:
system(sprintf("cp /content/saves_eod_%s.tar.gz %s/", today, DRIVE))

In [ ]:
file.remove("/content/saves_eod/mtl_sim.rds")
today   <- format(Sys.Date(), "%Y-%m-%d")
system(sprintf("cd /content && rm -f saves_eod_%s.tar.gz && tar -czf saves_eod_%s.tar.gz saves_eod/", today, today))
system(sprintf("cp /content/saves_eod_%s.tar.gz %s/", today, DRIVE))
cat("Re-saved:", round(file.info(sprintf("/content/saves_eod_%s.tar.gz", today))$size/1024^2, 2), "MB\n")
cat("Contents:"); print(list.files("/content/saves_eod"))

[1] TRUE

Re-saved: 32.85 MB
Contents:[1] "cma_age_data_mtlvan.rds" "fn_fit_stage1.rds"      
[3] "fn_qaic.rds"             "fn_reduce_fit.rds"      
[5] "mtl_substrate.rds"       "van_substrate.rds"      


In [ ]:
list.files(DRIVE, pattern = "saves_eod")

[1] "saves_eod_2026-06-10.tar.gz" "saves_eod_2026-06-17.tar.gz"